In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from torch.utils.data import Dataset, DataLoader
import torch

In [5]:
df = pd.read_csv('tokopedia-cleaned.csv', low_memory=False)
replace_map = {
    1 :'negative',
    2 : 'positive',
    0 : 'neutral'  
}
df['sentiment'] = df['sentiment'].replace(replace_map)
df.rename(columns={"content": "text"}, inplace=True)
df = df[["sentiment", "text"]]
df.head()

,sentiment,text
0,positive,Tokped mantaap
1,negative,kurir pengiriman nya sangat mengecewakan ID ex...
2,negative,Pembatalan belanja status masih proses butuh w...
3,negative,satu fitur yang sangat disayangkan hilang tanp...
4,positive,toped mantab


In [6]:
category_counts = df.groupby('sentiment').size()

# Memfilter kategori yang memiliki lebih dari satu teks
df = df[df['sentiment'].isin(category_counts[category_counts > 1].index)]

print('Total number of news: {}'.format(len(df)))
print(40*'-')
print('Split by sentiment:')

print(df["sentiment"].value_counts())
print(40*'-')
nr_categories = len(df["sentiment"].unique())

print("Number of sentiment: {n}".format(n=nr_categories))

Total number of news: 990
----------------------------------------
Split by sentiment:
sentiment
negative    484
positive    422
neutral      84
Name: count, dtype: int64
----------------------------------------
Number of sentiment: 3


In [7]:
# Split dataset
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['sentiment'].tolist(),
    test_size=0.2,
    stratify=df['sentiment'],
    random_state=42
)

# Map sentiment to numerical labels
label_map = {label: idx for idx, label in enumerate(sorted(df['sentiment'].unique()))}
train_labels = [label_map[label] for label in train_labels]
test_labels = [label_map[label] for label in test_labels]

In [8]:
# Load tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenization function
def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    return encodings, labels

# Tokenize data
train_encodings, train_labels = tokenize_data(train_texts, train_labels)
test_encodings, test_labels = tokenize_data(test_texts, test_labels)

In [9]:
# Create Dataset class
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)

In [10]:
# Load model
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=nr_categories)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [27]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    predictions, labels = pred
    predictions = predictions.argmax(axis=1)  # Ambil prediksi kelas dengan probabilitas tertinggi
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=3e-5,
    logging_dir='./logs',
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,  # Enable mixed precision
    metric_for_best_model="accuracy",
    greater_is_better=True
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# Trainer setup
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Train model
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Evaluate model
predictions = trainer.predict(test_dataset)
pred_labels = torch.argmax(torch.tensor(predictions.predictions), axis=1)
accuracy = accuracy_score(test_labels, pred_labels)

print("\nClassification Report:")
print(classification_report(test_labels, pred_labels, target_names=label_map.keys()))
print(f"Accuracy: {accuracy:.4f}")